In [5]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from transformers import AutoModel, AutoTokenizer

In [6]:
class MultiNERHead(nn.Module):
    """
    Creates multiple independent NER classifiers, each specialized in one category.
    """
    def __init__(self, hidden_size, num_ner_heads, num_labels_per_head):
        super(MultiNERHead, self).__init__()
        self.num_ner_heads = num_ner_heads
        self.ner_heads = nn.ModuleList([
            nn.Linear(hidden_size, num_labels_per_head) for _ in range(num_ner_heads)
        ])

    def forward(self, sequence_output):
        """
        Runs each NER head independently.
        Returns a list of token classification logits from each NER head.
        """
        ner_outputs = [head(sequence_output) for head in self.ner_heads]
        return ner_outputs  # List of (batch, seq_len, num_labels_per_head)


In [7]:
class AdvancedEntityAttention(nn.Module):
    """
    Attention mechanism that computes relationships between different NER heads.
    """
    def __init__(self, hidden_size):
        super(AdvancedEntityAttention, self).__init__()
        self.query_dense = nn.Linear(hidden_size, hidden_size)
        self.key_dense = nn.Linear(hidden_size, hidden_size)
        self.value_dense = nn.Linear(hidden_size, hidden_size)
        self.softmax = nn.Softmax(dim=-1)

    def forward(self, sequence_output, ner_outputs, entity_pairs):
        """
        Compute attention between different entity groups.
        
        Inputs:
        - sequence_output: (batch, seq_len, hidden_size)
        - ner_outputs: List of (batch, seq_len, num_labels_per_head)
        - entity_pairs: List of (head_index_1, head_index_2) defining attention pairs.

        Returns:
        - List of attention outputs for each entity pair.
        """
        batch_size, seq_len, hidden_dim = sequence_output.shape
        attention_outputs = {}

        for head_idx_1, head_idx_2 in entity_pairs:
            ner_logits_1 = ner_outputs[head_idx_1]
            ner_logits_2 = ner_outputs[head_idx_2]

            # Compute probability distributions for each NER head
            entity_probs_1 = F.softmax(ner_logits_1, dim=-1).sum(dim=-1, keepdim=True)  # (batch, seq_len, 1)
            entity_probs_2 = F.softmax(ner_logits_2, dim=-1).sum(dim=-1, keepdim=True)  # (batch, seq_len, 1)

            # Compute queries from entity 1, keys/values from entity 2
            queries = self.query_dense(sequence_output) * entity_probs_1
            keys = self.key_dense(sequence_output) * entity_probs_2
            values = self.value_dense(sequence_output) * entity_probs_2

            # Compute scaled dot-product attention
            scores = torch.matmul(queries, keys.transpose(-2, -1)) / (hidden_dim ** 0.5)
            attention_weights = self.softmax(scores)  # (batch, seq_len, seq_len)

            # Compute final attention output
            attention_output = torch.matmul(attention_weights, values)  # (batch, seq_len, hidden_size)
            attention_outputs[(head_idx_1, head_idx_2)] = attention_output

        return attention_outputs  # Dictionary of attention outputs per entity pair


In [8]:
class PrivacyDetectionModel(nn.Module):
    def __init__(self, base_model, num_ner_heads, num_labels_per_head, entity_pairs):
        """
        - base_model: ModernBERT encoder
        - num_ner_heads: Number of separate NER heads
        - num_labels_per_head: Number of labels per NER head
        - entity_pairs: List of entity pairs for attention learning
        """
        super(PrivacyDetectionModel, self).__init__()
        self.base_model = base_model
        hidden_size = base_model.config.hidden_size

        # Multiple NER heads
        self.multi_ner_head = MultiNERHead(hidden_size, num_ner_heads, num_labels_per_head)

        # Attention mechanism between NER groups
        self.entity_attention = AdvancedEntityAttention(hidden_size)

        # Store entity pairs for attention computation
        self.entity_pairs = entity_pairs

    def forward(self, input_ids, attention_mask):
        """
        Forward pass.
        - Extracts features from ModernBERT.
        - Runs multiple independent NER classifiers.
        - Computes attention between entity groups.

        Outputs:
        - CLS token representation.
        - Multiple NER logits.
        - Attention outputs between entity pairs.
        """
        outputs = self.base_model(input_ids=input_ids, attention_mask=attention_mask)
        sequence_output = outputs.last_hidden_state  # (batch, seq_len, hidden_size)
        cls_output = sequence_output[:, 0, :]  # Extract CLS token

        # Multiple NER head outputs
        ner_outputs = self.multi_ner_head(sequence_output)

        # Compute attention between selected entity pairs
        attention_outputs = self.entity_attention(sequence_output, ner_outputs, self.entity_pairs)

        return {
            "cls_output": cls_output,
            "ner_outputs": ner_outputs,
            "attention_outputs": attention_outputs
        }


In [9]:
model_name = "answerdotai/ModernBERT-base"  # Replace with an actual available model
tokenizer = AutoTokenizer.from_pretrained(model_name)
base_model = AutoModel.from_pretrained(model_name)

In [10]:
num_ner_heads = 2  # One for PERSON, one for PII
num_labels_per_head = 5  # Assuming BIO tagging per entity group
entity_pairs = [(0, 1)]  # Compute attention between PERSON (0) and PII (1)


In [11]:
model = PrivacyDetectionModel(base_model, num_ner_heads, num_labels_per_head, entity_pairs)


In [12]:
print(model)

PrivacyDetectionModel(
  (base_model): ModernBertModel(
    (embeddings): ModernBertEmbeddings(
      (tok_embeddings): Embedding(50368, 768, padding_idx=50283)
      (norm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
      (drop): Dropout(p=0.0, inplace=False)
    )
    (layers): ModuleList(
      (0): ModernBertEncoderLayer(
        (attn_norm): Identity()
        (attn): ModernBertAttention(
          (Wqkv): Linear(in_features=768, out_features=2304, bias=False)
          (rotary_emb): ModernBertRotaryEmbedding()
          (Wo): Linear(in_features=768, out_features=768, bias=False)
          (out_drop): Identity()
        )
        (mlp_norm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (mlp): ModernBertMLP(
          (Wi): Linear(in_features=768, out_features=2304, bias=False)
          (act): GELUActivation()
          (drop): Dropout(p=0.0, inplace=False)
          (Wo): Linear(in_features=1152, out_features=768, bias=False)
        )
      )
      (